To begin, I am importing all the necessary tools. I will need standard Python libraries for handling paths and system variables, as well as Gymnasium for creating the Atari Breakout environment. The core of the algorithmic part will be provided by stable-baselines3 and its PPO implementation. Additionally, I am connecting my own modules from the `package` package, which handle preprocessing, logging, and auxiliary services. This will allow me to keep the main notebook code clean and focused on the training process.
## Imports

In [ ]:
import os
import sys
import warnings
from functools import partial
from typing import Callable, Optional

In [ ]:
import gymnasium as gym
import ale_py
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
from stable_baselines3.common.atari_wrappers import FireResetEnv, EpisodicLifeEnv, ClipRewardEnv

In [ ]:
warnings.filterwarnings("ignore")
gym.register_envs(ale_py)
if os.path.abspath("package") not in sys.path: sys.path.append(os.path.abspath("package"))

In [ ]:
from package.environment import GymPreprocessing, create_breakout_env_gym
from package.dqn_types import ModelParameters, PathsParameters
from package.Logger import SmartLogger, LogsConfig
from package.sb3_utils import Callback, Checkpointer, EvaluateReward, VideoWriter, Support, is_notebook

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

At this stage, I am setting up the visual data processing pipeline. For Breakout, it is critically important to correctly prepare the frames: I am using `AtariPreprocessing` to resize them to 64x64 and perform frame skipping, while `FrameStackObservation` allows the agent to see the dynamics of the ball's movement by stacking the last 4 frames together. I am also adding specific wrappers: `EpisodicLifeEnv` for correct handling of life loss, and `FireResetEnv`, since in Breakout the game only starts after pressing the "FIRE" button. Finally, I am preparing functions to create both single and parallel environments (via `SubprocVecEnv`) to maximize CPU resource utilization during experience collection.
## Setup environment

In [ ]:
env_prep = GymPreprocessing(
    partial(AtariPreprocessing, noop_max=20, frame_skip=4, screen_size=64),
    partial(EpisodicLifeEnv),
    partial(FireResetEnv),
    partial(ClipRewardEnv),
    partial(FrameStackObservation, stack_size=4)
)

In [ ]:
build_env: Callable[[], gym.Env] = lambda: create_breakout_env_gym(transform=env_prep)
dummy_env: Callable[[int], DummyVecEnv] = lambda cnt: DummyVecEnv(list(build_env for _ in range(cnt)))
multi_env: Callable[[int], SubprocVecEnv] = lambda cnt: SubprocVecEnv(list(build_env for _ in range(cnt)))

Here I am defining the architecture and hyperparameters for my PPO model. I have chosen `CnnPolicy` as we are working with images. A learning rate of `lr=2.5e-4` is a proven standard for Atari tasks. I am also setting the entropy coefficient `ent_coef=0.01` so that the agent continues to explore the environment and doesn't get stuck in local minima, and I am limiting the `clip_range=0.1`, which makes strategy updates more conservative and stable. The `n_steps=128` parameter, combined with 8 parallel environments, gives me a batch of 1024 samples for each update iteration, providing a good balance between data collection speed and gradient accuracy.
## Model

In [ ]:
model_space: ModelParameters = ModelParameters(
    lr=2.5e-4,
    batch_size=256,
    n_epochs=8,
    n_steps=128,
    n_parallel=8,
    max_grad_norm=1.,
    n_frames=4
)

In [ ]:
envir = dummy_env(model_space.n_parallel) if is_notebook() else multi_env(model_space.n_parallel)

In [ ]:
model = PPO(
    "CnnPolicy",
    envir,
    verbose=0,
    learning_rate=model_space.lr,
    max_grad_norm=model_space.max_grad_norm,
    n_steps=model_space.n_steps,
    batch_size=model_space.batch_size,
    n_epochs=model_space.n_epochs,
    device=model_space.dev,
    ent_coef=0.01,
    clip_range=0.1,
    vf_coef=0.5,
)

To avoid losing progress and to be able to analyze the training, I am initializing the `SmartLogger`. It will be responsible for saving metrics, model weights, and gameplay recordings. I am setting the save frequency so that intermediate results are captured every 10,000 steps. This provides me with a detailed training history and allows me to visually assess how the agent's behavior changes at different stages.
## Logger

In [ ]:
paths_space = PathsParameters(exp_name="ppo", log_dir="breakout_logs")
logs_config = LogsConfig(paths_space.log_dir,
                         metrics_save_freq=int(1e3),
                         weights_save_freq=int(1e4),
                         videos_save_freq=int(1e4))
logger = SmartLogger(model.__class__.__name__, options=logs_config, exp_name=paths_space.exp_name)

For comprehensive monitoring, I need additional services. I am creating a `services` tuple that includes: `EvaluateReward` for periodic independent evaluation of gameplay quality (over 100 episodes), `Checkpointer` for automatic saving of the best model versions, and `VideoWriter` which will record game snippets. These assistants will be called automatically through my custom Callback without cluttering the main training loop.
## Services

In [ ]:
services: tuple[Support, ...] = (
    EvaluateReward(model, build_env(), freq=logger.options.weights_save_freq, n_episodes=100),
    Checkpointer(model, freq=logger.options.weights_save_freq, path=logger.model_paths[model.__class__.__name__]),
    VideoWriter(model, build_env, freq=logger.options.videos_save_freq, path=os.path.join(logger.path, "videos")),
)

Before starting the training, I am checking if there are already saved model states. If the logger finds a path to the latest checkpoint, I am loading it. This is a critical part of the process that allows me to interrupt and resume training without losing accumulated experience, which is especially relevant for long sessions of 10 million steps or more.
## Resuming train process

In [ ]:
last_upd: Optional[str] = logger.get_last_update(model.__class__.__name__)
model = model.load(last_upd, env=envir, device=model_space.dev) if last_upd else model

Finally, I am launching the main training process. I am calculating the total number of timesteps based on the number of iterations, environments, and steps per iteration. As a bridge between the PPO algorithm and my services, I am using a `Callback`, which will also render a convenient progress bar in the terminal. Now the agent starts playing, gathering experience, and gradually improving its strategy, aiming to score the maximum number of points in Breakout.
## Training

In [ ]:
total_timesteps: int = int(1e4) * model_space.n_parallel * model_space.n_steps
callback = Callback(*services, writer=logger, show_progress=total_timesteps // model_space.n_parallel)
model = model.learn(total_timesteps=total_timesteps, callback=callback)